# 2. From FASTQ to scores

`be_scan.analysis` counts reads, normalizes them, and compares conditions.

The example is a FOXA1 ABE sort screen: a `t0` reference, three `unsorted`
replicates, three `sorted-top`, three `sorted-bottom`. The FASTQ files here
are small simulated ones so the notebook runs in seconds.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.width", 220)

DATA = "data"
sample_sheet = pd.read_csv(f"{DATA}/FOXA1_sample_sheet.csv")
sample_sheet

,sample_id,fastq_file,condition,agg_conditions,counts_file,noncounts_file,stats_file
0,0,fastq/t0.fastq.gz,t0,t0,counts-t0.csv,noncounts-t0.csv,stats-t0.txt
1,1,fastq/unsorted-1.fastq.gz,unsorted-1,unsorted,counts-unsorted-1.csv,noncounts-unsorted-1.csv,stats-unsorted-1.txt
2,2,fastq/unsorted-2.fastq.gz,unsorted-2,unsorted,counts-unsorted-2.csv,noncounts-unsorted-2.csv,stats-unsorted-2.txt
3,3,fastq/unsorted-3.fastq.gz,unsorted-3,unsorted,counts-unsorted-3.csv,noncounts-unsorted-3.csv,stats-unsorted-3.txt
4,4,fastq/sorted-top-1.fastq.gz,sorted-top-1,sorted_top,counts-sorted-top-1.csv,noncounts-sorted-top-1.csv,stats-sorted-top-1.txt
5,5,fastq/sorted-top-2.fastq.gz,sorted-top-2,sorted_top,counts-sorted-top-2.csv,noncounts-sorted-top-2.csv,stats-sorted-top-2.txt
6,6,fastq/sorted-top-3.fastq.gz,sorted-top-3,sorted_top,counts-sorted-top-3.csv,noncounts-sorted-top-3.csv,stats-sorted-top-3.txt
7,7,fastq/sorted-bottom-1.fastq.gz,sorted-bottom-1,sorted_bottom,counts-sorted-bottom-1.csv,noncounts-sorted-bottom-1.csv,stats-sorted-bottom-1.txt
8,8,fastq/sorted-bottom-2.fastq.gz,sorted-bottom-2,sorted_bottom,counts-sorted-bottom-2.csv,noncounts-sorted-bottom-2.csv,stats-sorted-bottom-2.txt
9,9,fastq/sorted-bottom-3.fastq.gz,sorted-bottom-3,sorted_bottom,counts-sorted-bottom-3.csv,noncounts-sorted-bottom-3.csv,stats-sorted-bottom-3.txt


The sample sheet drives everything. Required columns:

| column | meaning |
|---|---|
| `fastq_file` | reads for this sample |
| `condition` | the sample's own name, unique per row |
| `agg_conditions` | which group it replicates; rows sharing a value are replicates |
| `counts_file`, `noncounts_file`, `stats_file` | per-sample outputs |

`condition` vs `agg_conditions` is the important distinction: `sorted-top-1`,
`-2` and `-3` are three conditions belonging to one `sorted_top` group, and
that grouping is what the replicate statistics are computed across.

## Step 1 — `count_reads`

Searches each read for `KEY`, takes the spacer that follows, and matches it
against the library. `guide_len` is new in 3.0.0 and defaults to 20.

In [2]:
from be_scan.analysis import count_reads

counts = count_reads(
    sample_sheet  = "FOXA1_sample_sheet.csv",
    annotated_lib = "FOXA1_library.csv",
    in_dir        = DATA,
    out_dir       = "output",
    KEY           = "CGAAACACC",   # end of the hU6 promoter
    KEY_REV       = "GTTTGAGA",    # start of the scaffold
    KEY_INTERVAL  = (0, 60),
    guide_len     = 20,
    return_df     = True, save_files = False,
)
counts.head()

Count reads completed


,sgRNA_ID,sgRNA_seq,AtoG_muttype,AtoG_pos,gene,t0,unsorted-1,unsorted-2,unsorted-3,sorted-top-1,sorted-top-2,sorted-top-3,sorted-bottom-1,sorted-bottom-2,sorted-bottom-3
0,NTCTRL_191,AAAACAGGACGATGTGCGGC,NT Control,-1.0,Control,14,13,10,9,14,9,10,13,12,12
1,NTCTRL_020,AAAACATCGACCGAAAGCGT,NT Control,-1.0,Control,8,11,11,15,14,10,14,18,12,7
2,NTCTRL_062,AAACCCTATGCCCAAATGAG,NT Control,-1.0,Control,14,16,10,8,14,7,9,9,13,7
3,NTCTRL_052,AAACGAGATCGAGAAAGGTA,NT Control,-1.0,Control,11,14,9,11,12,12,10,11,15,13
4,Dep_cntrl_45,AAACTTACCACCCAGAAGGA,Essential Control,-1.0,Essential splice site,12,13,11,13,13,13,10,7,13,14


## Step 2 — `log_transform`

Normalizes to reads per million, takes log2, subtracts the control, then
summarises each group of replicates.

For every condition with **more than one replicate** it now also reports
significance. Single-replicate conditions get none, because a p-value across
one measurement is not defined.

In [3]:
from be_scan.analysis import log_transform

lfc = log_transform(
    sample_sheet   = "FOXA1_sample_sheet.csv",
    library_counts = counts,
    controls       = ["t0"],
    in_dir         = DATA,
    save = False, return_df = True,
)
[c for c in lfc.columns if c.startswith("sorted_top")]

Merge and normalize completed


['sorted_top_LFCminusControl_avg',
 'sorted_top_LFCminusControl_stdev',
 'sorted_top_LFCminusControl_pval',
 'sorted_top_LFCminusControl_neglogp',
 'sorted_top_LFCminusControl_pval_mod',
 'sorted_top_LFCminusControl_neglogp_mod',
 'sorted_top_LFCminusControl_qval',
 'sorted_top_LFCminusControl_qval_mod']

### Which significance column to use

Six columns per multi-replicate condition:

| column | what it is |
|---|---|
| `_pval` | one-sample t-test of the replicate LFCs against 0 |
| `_neglogp` | −log10 of it, for volcano plots |
| `_pval_mod` | the same test with variance shrunk toward a prior fitted across the library |
| `_neglogp_mod` | −log10 of it |
| `_qval` | Benjamini-Hochberg FDR of `_pval` |
| `_qval_mod` | Benjamini-Hochberg FDR of `_pval_mod` |

**With three replicates, prefer the moderated columns.** The ordinary t-test
ranks largely on how lucky each guide's variance estimate happened to be, so
a guide whose replicates agree closely can outrank one with a far larger
effect.

In [4]:
C = "sorted_top_LFCminusControl"
summary = pd.DataFrame({
    "p < 0.05":  [(lfc[f"{C}_pval"] < 0.05).sum(), (lfc[f"{C}_pval_mod"] < 0.05).sum()],
    "q < 0.05":  [(lfc[f"{C}_qval"] < 0.05).sum(), (lfc[f"{C}_qval_mod"] < 0.05).sum()],
}, index=["ordinary t-test", "moderated (shrunk)"])
summary

,p < 0.05,q < 0.05
ordinary t-test,39,0
moderated (shrunk),74,46


In [5]:
# read a p-value alongside its effect size, never instead of it
cols = [f"{C}_avg", f"{C}_stdev", f"{C}_pval", f"{C}_pval_mod", f"{C}_qval_mod"]
lfc.nsmallest(8, f"{C}_pval_mod")[["sgRNA_ID"] + cols].round(4)

,sgRNA_ID,sorted_top_LFCminusControl_avg,sorted_top_LFCminusControl_stdev,sorted_top_LFCminusControl_pval,sorted_top_LFCminusControl_pval_mod,sorted_top_LFCminusControl_qval_mod
188,sgRNA_595,1.2766,0.1568,0.0050,0.0000,0.0002
108,sgRNA_856,1.2031,0.1956,0.0087,0.0000,0.0003
93,Dep_cntrl_69,1.5053,0.6606,0.0586,0.0000,0.0003
53,sgRNA_787,1.1640,0.3960,0.0365,0.0000,0.0005
20,sgRNA_361,1.0204,0.1492,0.0070,0.0000,0.0005
196,sgRNA_784,1.0332,0.2065,0.0131,0.0000,0.0005
150,sgRNA_19,-0.8975,0.1682,0.0115,0.0001,0.0016
61,sgRNA_639,-0.8121,0.0787,0.0031,0.0001,0.0029


Guides that are absent from every replicate have identical log-fold-changes
and therefore no variance at all. `scipy` reports p=0 for that, which would
be −log10(p) ≈ 308 and would sit at the top of every volcano plot, so those
rows are `NaN` instead.

In [6]:
n_nan = lfc[f"{C}_pval"].isna().sum()
print(f"{n_nan} guides have no usable variance and are NaN rather than p=0")

0 guides have no usable variance and are NaN rather than p=0


## Step 3 — `compare_conds`

Subtracts one group's average from another's, using a small csv of
`name, treatment, control` rows.

In [7]:
from be_scan.analysis import compare_conds

comparisons = pd.read_csv(f"{DATA}/comparisons_FOXA1.csv")
display(comparisons)

conds = compare_conds(
    comparisons = comparisons,
    avg_conds   = lfc,
    controls    = ["t0"],
    save = False, return_df = True,
)
conds[["sgRNA_ID", "sorted_top-unsorted", "sorted_bottom-unsorted"]].head()

,name,treatment,control
0,sorted_top-unsorted,sorted_top,unsorted
1,sorted_bottom-unsorted,sorted_bottom,unsorted


Compare conditions completed


,sgRNA_ID,sorted_top-unsorted,sorted_bottom-unsorted
0,NTCTRL_191,-0.023389,0.290606
1,NTCTRL_020,-0.022068,-0.023173
2,NTCTRL_062,-0.238075,-0.150062
3,NTCTRL_052,-0.040649,0.274605
4,Dep_cntrl_45,-0.104847,-0.117040


## Step 4 — `calc_controls`

Writes the mean and spread of the negative controls, which is what the
+/- 2 SD guide lines on the plots are drawn from.

In [8]:
from be_scan.analysis import calc_controls

calc_controls(
    conditions        = lfc,
    stats_comparisons = ["sorted_top", "sorted_bottom"],
    neg_ctrl_col      = "gene",
    neg_ctrl_conditions = ["Control"],
    out_dir  = "output",
    save = True, return_txt = True,
)

calc_controls outputed to output/stats.txt
For comparison sorted_top_LFCminusControl_avg

Mean is -0.032047527535971836

Standard deviation is 0.3303882754134837

Mean +- 2 standard deviations 0.6287290232909956 -0.6928240783629392

For comparison sorted_bottom_LFCminusControl_avg

Mean is 0.06716210028636639

Standard deviation is 0.34136938112833703

Mean +- 2 standard deviations 0.7499008625430404 -0.6155766619703077

Calculating controls completed


## All four at once — `batch_process`

```python
from be_scan.analysis import batch_process

batch_process(
    sample_sheet  = "data/FOXA1_sample_sheet.csv",
    annotated_lib = "data/FOXA1_library.csv",
    comparisons   = "data/comparisons_FOXA1.csv",
    file_dir      = "data",
    controls      = ["t0"],
    guide_len     = 20,
    neg_ctrl_col  = "gene",
    neg_ctrl_conditions = ["Control"],
    stats_comparisons   = ["sorted_top", "sorted_bottom"],
)
```

It runs `count_reads`, `log_transform`, `compare_conds` and `calc_controls`
in order, writing each intermediate file.